# Évaluation CL — DBSCAN (non-supervisé) — Dataset 3 PRONOSTIA — by_condition

| Champ | Valeur |
|-------|--------|
| **Modèle** | DBSCAN Detector (non-supervisé, clustering par densité) |
| **Dataset** | FEMTO PRONOSTIA — roulements à billes industriels |
| **Scénario** | by_condition : Condition 1 → Condition 2 → Condition 3 (3 tâches) |
| **Expérience** | exp_055 — voir experiments/exp_055_dbscan_pronostia_by_condition/config_snapshot.yaml |
| **Sprint** | 10 — S10-06 |

> **Modèle non-supervisé** : DBSCAN ne reçoit pas les labels à l'entraînement.  
> Score d'anomalie = distance au cluster le plus proche dans l'espace 2D réduit.  
> AF = 0.000 — aucun oubli grâce à l'accumulation de clusters entre conditions.  
> ⚠️ **RAM = 121 024 B (~118 Ko) > 64 Ko** — dépasse le budget STM32N6.  
> **Gap 1** : résultat CL non-supervisé sur données industrielles réelles de roulements (PRONOSTIA).

```bash
jupyter nbconvert --to notebook --execute \
    notebooks/cl_eval/pronostia_by_condition/dbscan.ipynb \
    --output /tmp/dbscan_pronostia_executed.ipynb --ExecutePreprocessor.timeout=300
```

In [ ]:
# Section 1 — Setup & imports
import json
import os
import pickle
import sys
from datetime import datetime
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

# --- CWD navigation : notebook 3 niveaux de profondeur ---
_cwd = Path(".").resolve()
if _cwd.name == "pronostia_by_condition":
    os.chdir(_cwd.parent.parent.parent)
elif _cwd.name == "cl_eval":
    os.chdir(_cwd.parent.parent)
elif _cwd.name == "notebooks":
    os.chdir(_cwd.parent)
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.plots import (
    plot_accuracy_matrix,
    plot_confusion_matrix_grid,
    plot_forgetting_curve,
    plot_roc_curves_per_task,
    save_figure,
)
from src.evaluation.feature_space_plots import plot_clustering_with_correctness

# --- Chemins ---
EXP_DIR         = REPO_ROOT / "experiments/exp_055_dbscan_pronostia_by_condition/results"
CHECKPOINTS_DIR = REPO_ROOT / "experiments/exp_055_dbscan_pronostia_by_condition/checkpoints"
FIGURES_DIR     = REPO_ROOT / "notebooks/figures/cl_evaluation/dbscan/pronostia/by_condition"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

NPY_DIR         = REPO_ROOT / "data/raw/Pronostia dataset/binaries"
NORMALIZER_PATH = REPO_ROOT / "configs/pronostia_normalizer.yaml"

# --- Constantes ---
TASK_NAMES    = ["Condition 1 (1800rpm, 4000N)", "Condition 2 (1650rpm, 4200N)", "Condition 3 (1500rpm, 5000N)"]
MODEL_NAME    = "DBSCAN"
DATASET_TAG   = "dataset3"
METRICS_FILE  = "metrics_all.json"
DATA_AVAILABLE = NPY_DIR.exists() and NORMALIZER_PATH.exists() and CHECKPOINTS_DIR.exists()

print(f"REPO_ROOT        : {REPO_ROOT}")
print(f"EXP_DIR          : {EXP_DIR}")
print(f"FIGURES_DIR      : {FIGURES_DIR}")
print(f"Data disponible  : {DATA_AVAILABLE}")
print(f"Date exécution   : {datetime.now():%Y-%m-%d %H:%M}")

In [ ]:
# Section 2 — Chargement des résultats exp_055

metrics_path = EXP_DIR / METRICS_FILE

raw = json.loads(metrics_path.read_text())
# Structure : metrics_all.json → clé "dbscan"
db  = raw["dbscan"]
mem = db["memory"]

acc_matrix_np = np.array(
    [[v if v is not None else np.nan for v in row] for row in db["acc_matrix"]],
    dtype=float,
)

BUDGET_64KO = 65536
ram_b = mem["ram_peak_bytes"]

print("=" * 55)
print(f"  Modèle   : DBSCAN (non-supervisé)")
print(f"  AA       = {db['aa']:.4f}")
print(f"  AF       = {db['af']:.4f}  ← zéro oubli (accumulation clusters)")
print(f"  BWT      = {db['bwt']:+.4f}")
print(f"  AUROC    = {db['auroc_avg']:.4f}")
print(f"  AUROC/tâche: {[round(v, 4) for v in db['auroc_per_task']]}")
print(f"  RAM      = {ram_b} B ({ram_b/1024:.1f} Ko)")
if ram_b > BUDGET_64KO:
    print(f"  ⚠️ FIXME(gap2) : RAM {ram_b/1024:.1f} Ko > 64 Ko — dépasse budget STM32N6")
print(f"  Latence  = {mem['inference_latency_ms']:.5f} ms")
print(f"  n_params = {mem['n_params']}")
print("=" * 55)
print("\nMatrice acc (3×3) :")
print(acc_matrix_np)

In [ ]:
# Section 3 — Matrice d'accuracy (heatmap)

fig = plot_accuracy_matrix(
    acc_matrix_np,
    task_names=TASK_NAMES,
    title=f"{MODEL_NAME} — pronostia/by_condition",
)
save_figure(fig, FIGURES_DIR / "acc_matrix.png")
plt.close(fig)
display(Image(str(FIGURES_DIR / "acc_matrix.png")))

In [ ]:
# Section 4 — Courbe d'oubli par tâche
# AF = 0.000 — DBSCAN accumule les clusters sans écraser les anciens

fig = plot_forgetting_curve(acc_matrix_np, task_names=TASK_NAMES)
save_figure(fig, FIGURES_DIR / "forgetting_curve.png")
plt.close(fig)
display(Image(str(FIGURES_DIR / "forgetting_curve.png")))

In [ ]:
# Section 5 — Inférence avec les checkpoints intermédiaires
# Charge dbscan_dataset3_task{0,1,2}.pkl

preds_dict  = {}
scores_dict = {}
X_tasks     = []
y_true_tasks = []
y_pred_tasks = []
cluster_tasks = []

if DATA_AVAILABLE:
    from sklearn.metrics import accuracy_score, roc_auc_score
    from src.data.pronostia_dataset import get_pronostia_dataloaders

    cl_tasks = get_pronostia_dataloaders(
        npy_dir=NPY_DIR,
        normalizer_path=NORMALIZER_PATH,
        batch_size=256,
        seed=42,
    )

    T = len(TASK_NAMES)

    val_data = []
    for task in cl_tasks:
        Xb, yb = [], []
        for X_batch, y_batch in task["val_loader"]:
            Xb.append(X_batch.numpy())
            yb.append(y_batch.numpy())
        val_data.append((np.concatenate(Xb), np.concatenate(yb).ravel()))

    for i in range(T):
        ckpt_path = CHECKPOINTS_DIR / f"dbscan_{DATASET_TAG}_task{i}.pkl"
        with open(ckpt_path, "rb") as f:
            dbscan_i = pickle.load(f)

        print(f"\nCheckpoint task{i} ({ckpt_path.name})")

        for j in range(i + 1):
            X_val, y_val = val_data[j]
            scores = dbscan_i.anomaly_score(X_val)
            y_pred = dbscan_i.predict(X_val)

            preds_dict[(i, j)]  = (y_val, y_pred)
            scores_dict[(i, j)] = scores

            acc_j   = accuracy_score(y_val, y_pred)
            auroc_j = roc_auc_score(y_val, scores) if len(np.unique(y_val)) >= 2 else float("nan")
            print(f"  Eval {TASK_NAMES[j][:15]:15s} | N={len(y_val):4d} | acc={acc_j:.4f} | AUROC={auroc_j:.4f}")

    with open(CHECKPOINTS_DIR / f"dbscan_{DATASET_TAG}_task{T - 1}.pkl", "rb") as f:
        dbscan_det = pickle.load(f)

    for j in range(T):
        X_val, y_val = val_data[j]
        X_tasks.append(X_val)
        y_true_tasks.append(y_val)
        y_pred_tasks.append(dbscan_det.predict(X_val))
        cluster_tasks.append(np.zeros(len(y_val), dtype=int))

    print(f"\npreds_dict clés : {sorted(preds_dict.keys())}")

else:
    display(Markdown("> ⚠️ **Mode dégradé** — données ou checkpoints absents. preds_dict synthétique."))

    N_SYNTH = 500
    rng = np.random.default_rng(42)
    y_synth = np.concatenate([np.zeros(int(N_SYNTH * 0.9)), np.ones(int(N_SYNTH * 0.1))])

    for i in range(3):
        for j in range(i + 1):
            noise = rng.normal(0, 0.15, len(y_synth))
            scores_s = np.where(y_synth == 1, 0.72 + noise, 0.28 + noise).clip(0, 1)
            y_pred_s = (scores_s >= 0.5).astype(float)
            preds_dict[(i, j)]  = (y_synth.copy(), y_pred_s)
            scores_dict[(i, j)] = scores_s.astype(np.float32)

    print("preds_dict synthétique créé (mode dégradé)")

In [ ]:
# Section 6 — Matrices de confusion par tâche

fig = plot_confusion_matrix_grid(
    preds_dict,
    task_names=TASK_NAMES,
    model_name=MODEL_NAME,
)
save_figure(fig, FIGURES_DIR / "confusion_matrix_grid.png")
plt.close(fig)
display(Image(str(FIGURES_DIR / "confusion_matrix_grid.png")))

In [ ]:
# Section 7 — Courbes AUROC par tâche

fig = plot_roc_curves_per_task(
    preds_dict,
    scores_dict=scores_dict,
    task_names=TASK_NAMES,
    model_name=MODEL_NAME,
)
save_figure(fig, FIGURES_DIR / "roc_curves.png")
plt.close(fig)
display(Image(str(FIGURES_DIR / "roc_curves.png")))

In [ ]:
# Section 8 — Espace des features PCA 2D

if DATA_AVAILABLE and len(X_tasks) == 3:
    fig = plot_clustering_with_correctness(
        X_tasks,
        y_true_tasks,
        y_pred_tasks,
        cluster_tasks,
        task_names=TASK_NAMES,
        model_name=MODEL_NAME,
        projection="pca",
    )
    save_figure(fig, FIGURES_DIR / "feature_space_pca.png")
    plt.close(fig)
    display(Image(str(FIGURES_DIR / "feature_space_pca.png")))

else:
    display(Markdown(
        "> ⚠️ **Section 8 ignorée** — données ou checkpoints absents. "
        "feature_space_pca.png non généré."
    ))
    print("[SKIP] feature_space_pca.png — données non disponibles.")

## Discussion — Gap 1

Ces résultats non-supervisés sur FEMTO PRONOSTIA (données industrielles réelles de roulements à billes)
complètent les validations sur les datasets Kaggle.

**Comparaison cross-dataset** :
- Dataset Monitoring (by_equipment, 3 tâches) : AA ≈ ?, AUROC ≈ ?
- **Dataset PRONOSTIA (by_condition, 3 tâches) : AA = 0.9014, AUROC = 0.8250** ← exp_055

**Zéro oubli (AF = 0.000)** :
DBSCAN accumule les clusters de toutes les conditions vues — il ne réinitialise pas.
Cette propriété est avantageuse pour la stabilité CL mais a un coût en RAM qui croît
linéairement avec le nombre de tâches.

**Contrainte embarquée** :
`FIXME(gap2)` : DBSCAN RAM = 121 024 B (~118 Ko) dépasse le budget STM32N6 (64 Ko).
À compresser (quantification INT8 des données DBSCAN) ou remplacer par KMeans pour le déploiement MCU.

`FIXME(gap1)` : ✅ Résultat CL non-supervisé (DBSCAN) sur données industrielles réelles de
roulements — voir `docs/roadmap_phase1.md` section Sprint 10 pour la synthèse complète.

`TODO(arnaud)` : La section Discussion Gap 1 doit-elle inclure une référence bibliographique
au protocole IEEE PHM 2012 Challenge (`@PHM2012`) pour contextualiser dans la littérature ?

In [ ]:
# Section 10 — Tableau récapitulatif + critères d'acceptation

aa    = db["aa"]
af    = db["af"]
bwt   = db["bwt"]
auroc = db["auroc_avg"]
ram_b = mem["ram_peak_bytes"]
ram_ko = ram_b / 1024
lat   = mem["inference_latency_ms"]
n_par = mem["n_params"]

display(Markdown("### Résultats finaux — DBSCAN — pronostia/by_condition (exp_055)"))

recap_table = f"""
| Modèle | AA ↑ | AF ↓ | BWT | AUROC ↑ | RAM ↓ | Latence ↓ | n_params |
|--------|------|------|-----|---------|-------|-----------|----------|
| {MODEL_NAME} | {aa:.4f} | {af:.4f} | {bwt:+.4f} | {auroc:.4f} | {ram_ko:.1f} Ko ⚠️ | {lat:.5f} ms | {n_par} |
"""
display(Markdown(recap_table))

print("=" * 60)
print("  NOTE — FIXME(gap2) : contrainte RAM STM32N6")
print("=" * 60)
print(f"  RAM DBSCAN = {ram_b} B = {ram_ko:.1f} Ko")
print(f"  Budget     = {BUDGET_64KO} B = 64 Ko")
print(f"  Dépassement= {ram_b - BUDGET_64KO} B ({(ram_b - BUDGET_64KO)/1024:.1f} Ko au-delà)")
print(f"  → DBSCAN non portables tel quel sur STM32N6")
print()

print("=" * 60)
print("  Critères d'acceptation (S10-06)")
print("=" * 60)
for fig_name in ["acc_matrix.png", "forgetting_curve.png", "confusion_matrix_grid.png",
                 "roc_curves.png", "feature_space_pca.png"]:
    status = "OK" if (FIGURES_DIR / fig_name).exists() else "MANQUANTE"
    print(f"  [{status}] {fig_name}")

print()
print(f"  [{'OK' if abs(aa - 0.9014) < 0.005 else 'WARN'}] AA     = {aa:.4f}  (attendu ≈ 0.9014)")
print(f"  [{'OK' if af < 0.005 else 'WARN'}] AF     = {af:.4f}  (attendu ≈ 0.0000)")
print(f"  [{'OK' if abs(auroc - 0.8250) < 0.005 else 'WARN'}] AUROC  = {auroc:.4f}  (attendu ≈ 0.8250)")
print(f"  [{'FAIL' if ram_b > BUDGET_64KO else 'OK'}] RAM    = {ram_ko:.1f} Ko (contrainte ≤ 64 Ko) ← FIXME(gap2)")
print(f"  [{'OK' if lat < 100.0 else 'WARN'}] Latence= {lat:.5f} ms (contrainte ≤ 100 ms)")

figs = sorted(FIGURES_DIR.glob("*.png"))
print(f"\nFigures générées ({len(figs)}/5) : {[f.name for f in figs]}")